# Stage 13B.5 — Matched-Control Recovery, Spatial Safeguards and Provisional Dataset

**Goal:** reconcile the 33 control-only failure groups using the Stage 13B.4A feature-recovered candidates. The 121 original valid matched groups remain untouched. **This notebook does not overwrite Stage 06–12 datasets or models, and does not claim final scientific approval.**

### Critical safeguards
- Check original sample identity against Stage 05B matched groups and Stage 05A static-valid candidate pool.
- Independently check global control uniqueness and nearest historical case distances.
- Original Stage 04 pseudo-absence *exclusion distance/polygon* is not encoded in Stage 13B.3's radius ranking. Unless that rule is supplied and verified, **spatial approval remains PENDING**.
- Dynamic World recovered classifications use a strictly pre-event window and >=3 observations; existing annual land cover needs a historical-leakage assessment.
- Export a **provisional** 1:2 group reconstruction separately from any approved model-ready dataset.
- Never use model performance to select controls.

## PART A — INPUTS AND CONFIGURATION
### 1. Configure paths and original spatial safeguards

In [13]:
from pathlib import Path
import json
import math
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(r"C:\Landslide_FYP")
DEV_DIR = PROJECT_ROOT / "Landslide_FYP_Development"
RESULT_DIR = DEV_DIR / "05_model_results"

CANDIDATE_PATH = RESULT_DIR / "stage13b4a_landcover_recovery" / "replacement_candidates_after_landcover_recovery.csv"
SLOTS_PATH = RESULT_DIR / "stage13b3_control_recovery" / "control_replacement_slots.csv"
STATIC_PATH = DEV_DIR / "03_derived_features" / "static_features_final" / "static_feature_samples_final.csv"
MASTER_PATH = DEV_DIR / "03_derived_features" / "temporal_features_final" / "temporal_feature_samples_final_annotated.csv"
ORIGINAL_FULL_PATH = DEV_DIR / "03_derived_features" / "temporal_features_final" / "temporal_model_full_enhanced.csv"
OUTPUT_DIR = RESULT_DIR / "stage13b5_final_matched_control_validation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# IMPORTANT: Find the exact number/geometry in the original Stage 04/05A pseudo-absence code.
# DO NOT guess it. A None value means spatial eligibility cannot be certified.
ORIGINAL_EXCLUSION_BUFFER_KM = None
ORIGINAL_BUFFER_SOURCE_NOTE = "NOT VERIFIED: inspect Stage 04/05A pseudo-absence generation code"

# Annual pre-existing GLC layers may contain observations after event dates.
# Retain provisional groups but do not authorize final retraining until reviewed.
ANNUAL_LANDCOVER_LEAKAGE_CLEARED = False

for title, path in {
    "recovered candidates": CANDIDATE_PATH,
    "required control slots": SLOTS_PATH,
    "static QC inventory": STATIC_PATH,
    "original temporal matched master": MASTER_PATH,
    "original full temporal subset": ORIGINAL_FULL_PATH,
}.items():
    print(title, path.exists(), "->", path)
    if not path.exists():
        raise FileNotFoundError(f"Missing {title}: {path}")
print("Output:", OUTPUT_DIR)

recovered candidates True -> C:\Landslide_FYP\Landslide_FYP_Development\05_model_results\stage13b4a_landcover_recovery\replacement_candidates_after_landcover_recovery.csv
required control slots True -> C:\Landslide_FYP\Landslide_FYP_Development\05_model_results\stage13b3_control_recovery\control_replacement_slots.csv
static QC inventory True -> C:\Landslide_FYP\Landslide_FYP_Development\03_derived_features\static_features_final\static_feature_samples_final.csv
original temporal matched master True -> C:\Landslide_FYP\Landslide_FYP_Development\03_derived_features\temporal_features_final\temporal_feature_samples_final_annotated.csv
original full temporal subset True -> C:\Landslide_FYP\Landslide_FYP_Development\03_derived_features\temporal_features_final\temporal_model_full_enhanced.csv
Output: C:\Landslide_FYP\Landslide_FYP_Development\05_model_results\stage13b5_final_matched_control_validation


### 2. Load and validate source schemas

In [14]:
candidates = pd.read_csv(CANDIDATE_PATH, low_memory=False)
slots = pd.read_csv(SLOTS_PATH, low_memory=False)
static = pd.read_csv(STATIC_PATH, low_memory=False)
master = pd.read_csv(MASTER_PATH, low_memory=False)
original_full = pd.read_csv(ORIGINAL_FULL_PATH, low_memory=False)

required_candidates = ["matched_positive_id", "old_control_sample_id", "sample_id", "reference_date", "longitude", "latitude", "pre_event_feature_valid_candidate", "revised_landcover_class_candidate", "revised_landcover_source_candidate"]
required_master = ["matched_positive_id", "sample_id", "temporal_role", "reference_date", "longitude", "latitude", "full_temporal_row_valid"]
for name, frame, required in [("candidates", candidates, required_candidates), ("master", master, required_master), ("slots", slots, ["matched_positive_id", "old_control_sample_id"]), ("static", static, ["sample_id", "sample_class", "national_core_static_valid_final", "longitude", "latitude"])]:
    missing = sorted(set(required) - set(frame.columns))
    if missing:
        raise ValueError(f"{name} missing required fields: {missing}")

for frame in (candidates, slots, static, master, original_full):
    for col in ("sample_id", "matched_positive_id", "old_control_sample_id"):
        if col in frame.columns:
            frame[col] = frame[col].astype("string").str.strip()
    if "reference_date" in frame.columns:
        frame["reference_date"] = pd.to_datetime(frame["reference_date"], errors="coerce").dt.normalize()

if candidates["reference_date"].isna().any():
    raise ValueError("Candidate historical dates cannot be null")
if static["sample_id"].duplicated().any():
    raise ValueError("Static inventory has duplicate sample IDs")
if master["sample_id"].duplicated().any():
    raise ValueError("Original temporal master has duplicate sample IDs")
if slots.duplicated(["matched_positive_id", "old_control_sample_id"]).any():
    raise ValueError("Duplicated control slot keys")

print("Candidate rows:", len(candidates))
print("Control slots:", len(slots), "in", slots["matched_positive_id"].nunique(), "groups")
print("Original full cases:", int(original_full["temporal_role"].eq("case").sum()))
print("Original full controls:", int(original_full["temporal_role"].eq("matched_control").sum()))

Candidate rows: 170
Control slots: 34 in 33 groups
Original full cases: 121
Original full controls: 242


## PART B — SPATIAL AND FEATURE CHECKS
### 3. Candidate eligibility and original-static consistency

In [15]:
def bool_values(series):
    return series.astype("string").str.strip().str.lower().isin(["true", "1", "yes"])

flags = ["rainfall_valid", "soil_moisture_complete", "waterway_available"]
for flag in flags:
    if flag not in candidates.columns:
        raise KeyError(f"Required original Stage 05B flag absent: {flag}")

static_ids = static[["sample_id", "sample_class", "national_core_static_valid_final", "longitude", "latitude"]].rename(columns={"longitude": "static_longitude", "latitude": "static_latitude"})
audit = candidates.merge(static_ids, on="sample_id", how="left", validate="many_to_one")
audit["static_valid_source"] = audit["sample_class"].eq("background_candidate") & bool_values(audit["national_core_static_valid_final"])
for col in ["longitude", "latitude", "static_longitude", "static_latitude"]:
    audit[col] = pd.to_numeric(audit[col], errors="coerce")
audit["coordinate_matches_static"] = (
    np.isclose(audit["longitude"], audit["static_longitude"], atol=0.000001, rtol=0, equal_nan=False)
    & np.isclose(audit["latitude"], audit["static_latitude"], atol=0.000001, rtol=0, equal_nan=False)
)
audit["pre_event_feature_pass"] = bool_values(audit["pre_event_feature_valid_candidate"])
audit["rechecked_numeric_features"] = np.logical_and.reduce([bool_values(audit[col]) for col in flags])
audit["nonwater_historical_class"] = audit["revised_landcover_class_candidate"].notna() & ~audit["revised_landcover_class_candidate"].eq("WATER")
audit["minimum_dw_observations"] = (
    ~audit["revised_landcover_source_candidate"].eq("Dynamic_World_PRE_EVENT_365D")
    | pd.to_numeric(audit.get("dw_valid_obs", pd.Series(np.nan, index=audit.index)), errors="coerce").ge(3)
)
audit["not_original_matched_sample"] = ~audit["sample_id"].isin(master["sample_id"])
audit["valid_for_provisional_selection"] = (
    audit["static_valid_source"] & audit["coordinate_matches_static"]
    & audit["pre_event_feature_pass"] & audit["rechecked_numeric_features"]
    & audit["nonwater_historical_class"] & audit["minimum_dw_observations"]
    & audit["not_original_matched_sample"]
)
print("Feature and pool eligible:", int(audit["valid_for_provisional_selection"].sum()), "/", len(audit))

Feature and pool eligible: 132 / 170


### 4. Nearest historical landslide distance (independent audit; not a guessed buffer)

In [16]:
def haversine_km(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(lambda x: np.radians(np.asarray(x, dtype=float)), [lon1, lat1, lon2, lat2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371.0088 * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

# Check against ALL coordinate-eligible positive samples, not merely the matched case.
positive_pool = static.loc[
    static["sample_class"].eq("landslide")
    | static["sample_id"].astype("string").str.startswith("POS_")
].copy()
positive_pool["longitude"] = pd.to_numeric(positive_pool["longitude"], errors="coerce")
positive_pool["latitude"] = pd.to_numeric(positive_pool["latitude"], errors="coerce")
positive_pool = positive_pool.dropna(subset=["longitude", "latitude"])
if positive_pool.empty:
    raise ValueError("No positive locations found for spatial exclusion audit")

# Calculate on unique points to reduce work.
locs = audit[["sample_id", "longitude", "latitude"]].drop_duplicates("sample_id")
nearest = []
for row in locs.itertuples(index=False):
    d = haversine_km(row.longitude, row.latitude, positive_pool["longitude"].to_numpy(), positive_pool["latitude"].to_numpy())
    i = int(np.argmin(d))
    nearest.append({"sample_id": row.sample_id, "nearest_positive_km": float(d[i]), "nearest_positive_id": str(positive_pool.iloc[i]["sample_id"])})
audit = audit.merge(pd.DataFrame(nearest), on="sample_id", how="left", validate="many_to_one")

if ORIGINAL_EXCLUSION_BUFFER_KM is None:
    audit["original_buffer_verified"] = False
    audit["buffer_screen_status"] = "PENDING_ORIGINAL_RULE"
else:
    if not (isinstance(ORIGINAL_EXCLUSION_BUFFER_KM, (int, float)) and ORIGINAL_EXCLUSION_BUFFER_KM > 0):
        raise ValueError("Original buffer must be a verified positive distance in km")
    audit["original_buffer_verified"] = True
    audit["buffer_screen_status"] = np.where(audit["nearest_positive_km"].ge(ORIGINAL_EXCLUSION_BUFFER_KM), "PASS_DISTANCE_ONLY", "FAIL_TOO_CLOSE")

# Membership in prior vetted background pool DOES NOT prove that a new matching design
# follows the original spatial exclusion/buffer logic; retain original-rule review gate.
audit["original_spatial_rule_approved"] = False
print("Nearest positive distance (km):")
print(audit["nearest_positive_km"].describe())
print("Original spatial approval remains pending until the original Stage 04/05A rules are verified.")

Nearest positive distance (km):
count    170.000000
mean      10.333670
std        5.177880
min        2.708479
25%        6.131707
50%        8.929050
75%       13.747037
max       22.186529
Name: nearest_positive_km, dtype: float64
Original spatial approval remains pending until the original Stage 04/05A rules are verified.


### 5. Date, slot and structural checks

In [17]:
group_ref = master.loc[master["temporal_role"].eq("case"), ["matched_positive_id", "reference_date", "longitude", "latitude", "full_temporal_row_valid"]].rename(columns={"reference_date": "case_reference_date", "longitude": "case_longitude", "latitude": "case_latitude", "full_temporal_row_valid": "case_original_qc_valid"})
if group_ref["matched_positive_id"].duplicated().any():
    raise ValueError("Original master has multiple cases per group")
audit = audit.merge(group_ref, on="matched_positive_id", how="left", validate="many_to_one")
audit["date_matches_case"] = audit["reference_date"].eq(audit["case_reference_date"])
audit["target_case_valid"] = bool_values(audit["case_original_qc_valid"])
slot_keys = slots[["matched_positive_id", "old_control_sample_id"]].drop_duplicates()
audit = audit.merge(slot_keys.assign(slot_required=True), on=["matched_positive_id", "old_control_sample_id"], how="left", validate="many_to_one")
audit["slot_required"] = audit["slot_required"].fillna(False).astype(bool)
audit["selection_eligible"] = audit["valid_for_provisional_selection"] & audit["date_matches_case"] & audit["target_case_valid"] & audit["slot_required"]

# Existing full groups must be untouched.
if len(original_full) != 363 or int(original_full["temporal_role"].eq("case").sum()) != 121:
    raise ValueError("Original full dataset differs from the verified 121 + 242 baseline; stop and reconcile")
if original_full["sample_id"].duplicated().any():
    raise ValueError("Original full dataset has duplicated sample IDs")
print("Candidates passing provisional selection checks:", int(audit["selection_eligible"].sum()))
print("Eligible slots:", audit.loc[audit["selection_eligible"], ["matched_positive_id", "old_control_sample_id"]].drop_duplicates().shape[0])

Candidates passing provisional selection checks: 132
Eligible slots: 33


## PART C — GLOBAL UNIQUE CONTROL ASSIGNMENT
### 6. Search for a maximum number of complete groups, with no reused background ID

In [18]:
# Exhaustive group-level combinations (up to 5 candidates per slot), then backtrack across
# groups to avoid reusing the same background location globally.
from itertools import product

pool = audit.loc[audit["selection_eligible"]].copy()
pool["candidate_rank"] = pd.to_numeric(pool.get("candidate_rank", pd.Series(999, index=pool.index)), errors="coerce").fillna(999)
pool = pool.sort_values(["candidate_rank", "nearest_positive_km", "sample_id"])
slot_list = [tuple(x) for x in slots[["matched_positive_id", "old_control_sample_id"]].itertuples(index=False, name=None)]
option_ids = {}
for group_id, old_id in slot_list:
    frame = pool.loc[pool["matched_positive_id"].eq(group_id) & pool["old_control_sample_id"].eq(old_id)]
    option_ids[(group_id, old_id)] = frame["sample_id"].astype(str).drop_duplicates().tolist()

by_group = {}
for group_id in slots["matched_positive_id"].drop_duplicates().astype(str):
    group_slots = [s for s in slot_list if s[0] == group_id]
    choices = [option_ids[s] for s in group_slots]
    combinations = []
    if all(choices):
        for combo in product(*choices):
            if len(set(combo)) == len(combo):
                combinations.append(tuple(zip(group_slots, combo)))
    by_group[group_id] = combinations

# Find the largest conflict-free set. Ordering by fewest choices reduces backtracking.
ordered = sorted(by_group, key=lambda g: (len(by_group[g]) == 0, len(by_group[g]), g))
best = {}
def search(i, used, chosen):
    global best
    if len(chosen) > len(best):
        best = dict(chosen)
    if i == len(ordered) or len(chosen) + len(ordered) - i <= len(best):
        return
    group = ordered[i]
    for comb in by_group[group]:
        ids = {candidate_id for _, candidate_id in comb}
        if not ids.intersection(used):
            chosen[group] = comb
            search(i + 1, used | ids, chosen)
            del chosen[group]
    search(i + 1, used, chosen)

search(0, set(master["sample_id"].dropna().astype(str)), {})
assigned = []
for group, comb in sorted(best.items()):
    for (grp, old_id), new_id in comb:
        assigned.append({"matched_positive_id": grp, "old_control_sample_id": old_id, "sample_id": new_id})
assignments = pd.DataFrame(assigned, columns=["matched_positive_id", "old_control_sample_id", "sample_id"])
if assignments["sample_id"].duplicated().any():
    raise AssertionError("Duplicate candidate used across matched groups")
print("Provisional complete recovery groups:", len(best))
print("Provisional replacement controls:", len(assignments))
print("Groups without complete unique assignment:", len(by_group) - len(best))

KeyboardInterrupt: 

### 7. Export assignment audit and unresolved candidate slots

In [ ]:
assignment_detail = assignments.merge(
    audit, on=["matched_positive_id", "old_control_sample_id", "sample_id"], how="left", validate="one_to_one"
)
assignment_detail["final_spatial_approved"] = assignment_detail["original_spatial_rule_approved"]
assignment_detail["final_temporal_policy_approved"] = ANNUAL_LANDCOVER_LEAKAGE_CLEARED
assignment_detail["final_replacement_approved"] = (
    assignment_detail["final_spatial_approved"] & assignment_detail["final_temporal_policy_approved"]
)
assignment_detail.to_csv(OUTPUT_DIR / "provisional_unique_control_assignments.csv", index=False)
audit.to_csv(OUTPUT_DIR / "candidate_spatial_and_feature_audit.csv", index=False)

slot_status = slots[["matched_positive_id", "old_control_sample_id"]].drop_duplicates().copy()
slot_status["eligible_options"] = slot_status.apply(
    lambda r: len(option_ids.get((str(r["matched_positive_id"]), str(r["old_control_sample_id"])), [])), axis=1
)
slot_status = slot_status.merge(
    assignments.assign(assigned=True)[["matched_positive_id", "old_control_sample_id", "sample_id", "assigned"]],
    on=["matched_positive_id", "old_control_sample_id"], how="left", validate="one_to_one"
)
slot_status["assigned"] = slot_status["assigned"].fillna(False).astype(bool)
slot_status.to_csv(OUTPUT_DIR / "replacement_slot_final_screen.csv", index=False)
print(slot_status.groupby("assigned").size())

## PART D — CONSTRUCT A PROVISIONAL MATCHED DATASET
### 8. Rebuild complete groups while retaining exact Stage 05B target schema

In [ ]:
# Preserve existing full-temporal rows. Keep the original 1 case and any valid old
# matched control for each recoverable group; replace only failing control slots.
needed_flags = ["rainfall_valid", "soil_moisture_complete", "landcover_valid_strict", "waterway_available"]
for col in needed_flags:
    if col not in master.columns:
        raise KeyError(f"Required original QC flag absent from annotated master: {col}")

master_valid = pd.concat([bool_values(master[c]).rename(c) for c in needed_flags], axis=1).all(axis=1)
master = master.assign(_recomputed_row_pass=master_valid)

def recovered_control_row(selected):
    # Static predictor values from the ORIGINAL candidate inventory, temporal predictors
    # from date-specific recovered extraction; copy the Stage 05B column names explicitly.
    original_static = static.loc[static["sample_id"].eq(selected["sample_id"])]
    if len(original_static) != 1:
        raise ValueError("Static source identity mismatch")
    rec = {col: pd.NA for col in master.columns}
    src = original_static.iloc[0]
    for col in master.columns:
        if col in src.index:
            rec[col] = src[col]
    for col in master.columns:
        if col in selected.index:
            rec[col] = selected[col]
    rec["sample_id"] = selected["sample_id"]
    rec["matched_positive_id"] = selected["matched_positive_id"]
    rec["temporal_role"] = "matched_control"
    rec["reference_date"] = selected["reference_date"]
    rec["landcover_harmonised"] = selected["revised_landcover_class_candidate"]
    rec["landcover_source"] = selected["revised_landcover_source_candidate"]
    rec["landcover_reliability"] = "HIGH" if selected["revised_landcover_source_candidate"] == "GLC_FCS30D" else "PRE_EVENT_DW_3PLUS_PROVISIONAL"
    for c in needed_flags:
        rec[c] = True
    rec["water_background_invalid"] = False
    rec["full_temporal_row_valid"] = True
    rec["_recomputed_row_pass"] = True
    return rec

recovered_groups = []
for group in sorted(best):
    part = master.loc[master["matched_positive_id"].eq(group)].copy()
    if len(part) != 3 or part["temporal_role"].eq("case").sum() != 1 or part["temporal_role"].eq("matched_control").sum() != 2:
        raise ValueError(f"Invalid original matched group {group}")
    to_replace = set(assignments.loc[assignments["matched_positive_id"].eq(group), "old_control_sample_id"])
    remaining = part.loc[~part["sample_id"].isin(to_replace)].copy()
    assert remaining["_recomputed_row_pass"].all(), f"Remaining row failed QC for {group}"
    new_rows = [recovered_control_row(a) for _, a in assignment_detail.loc[assignment_detail["matched_positive_id"].eq(group)].iterrows()]
    created = pd.DataFrame(new_rows).reindex(columns=master.columns)
    reconstructed = pd.concat([remaining, created], ignore_index=True)
    if not (len(reconstructed) == 3 and reconstructed["temporal_role"].eq("case").sum() == 1 and reconstructed["temporal_role"].eq("matched_control").sum() == 2):
        raise AssertionError(f"New group invalid: {group}")
    if reconstructed["sample_id"].nunique() != 3 or reconstructed["reference_date"].nunique() != 1:
        raise AssertionError(f"Duplicate IDs or mismatched dates: {group}")
    recovered_groups.append(reconstructed)

provisional = pd.concat([original_full.reindex(columns=master.columns)] + recovered_groups, ignore_index=True)
provisional = provisional.drop(columns=["_recomputed_row_pass"], errors="ignore")
if provisional["sample_id"].duplicated().any():
    raise AssertionError("Provisional dataset reuses an already-assigned sample")
if not provisional.groupby("matched_positive_id").size().eq(3).all():
    raise AssertionError("At least one provisional group does not have 3 samples")

# PROVISIONAL ONLY: explicitly retain this name to prevent confusion with the final training file.
provisional.to_csv(OUTPUT_DIR / "PROVISIONAL_recovered_temporal_matched_groups_NOT_FOR_TRAINING.csv", index=False)
print("Provisional positive cases:", int(provisional["temporal_role"].eq("case").sum()))
print("Provisional controls:", int(provisional["temporal_role"].eq("matched_control").sum()))
print("Original cases retained:", 121)
print("Additional provisional groups:", len(best))

### 9. Data integrity summary and unresolved scientific gates

In [ ]:
is_original_unchanged = original_full["sample_id"].astype("string").isin(provisional["sample_id"].astype("string")).all()
new_sources = assignment_detail["revised_landcover_source_candidate"].value_counts(dropna=False).to_dict() if len(assignment_detail) else {}
summary = {
    "original_valid_positive_cases": 121,
    "target_additional_control_only_groups": int(slots["matched_positive_id"].nunique()),
    "control_slots": len(slots),
    "provisional_complete_unique_groups": len(best),
    "provisional_replaced_controls": len(assignments),
    "unresolved_target_groups": int(slots["matched_positive_id"].nunique()) - len(best),
    "provisional_total_cases": int(provisional["temporal_role"].eq("case").sum()),
    "provisional_total_controls": int(provisional["temporal_role"].eq("matched_control").sum()),
    "original_case_ids_retained": bool(is_original_unchanged),
    "replacement_landcover_sources": {str(k): int(v) for k, v in new_sources.items()},
    "original_buffer_rule_verified": ORIGINAL_EXCLUSION_BUFFER_KM is not None,
    "original_spatial_rule_independently_approved": False,
    "pre_event_annual_landcover_leakage_cleared": ANNUAL_LANDCOVER_LEAKAGE_CLEARED,
    "globally_unique_candidate_assignments": not bool(assignments["sample_id"].duplicated().any()),
    "verified_recovered_groups": 0,
    "retraining_authorized": False,
    "note": "Provisional dataset is strictly not for model training until the original spatial exclusion policy and historical land-cover timing are cleared, then full nested spatial benchmarking is rerun."
}
(OUTPUT_DIR / "stage13b5_summary.json").write_text(json.dumps(summary, indent=2, default=str), encoding="utf8")
gate = pd.DataFrame([
    {"check": "Original full subset preserved", "status": "PASS" if is_original_unchanged else "FAIL", "details": "121 original cases plus matched controls"},
    {"check": "Global replacement uniqueness", "status": "PASS" if summary["globally_unique_candidate_assignments"] else "FAIL", "details": len(assignments)},
    {"check": "Feature/date/matched-group provisional validation", "status": "PROVISIONAL_PASS", "details": len(best)},
    {"check": "Original pseudo-absence spatial rule", "status": "PENDING", "details": ORIGINAL_BUFFER_SOURCE_NOTE},
    {"check": "Annual historical land-cover temporal leakage", "status": "REVIEW_REQUIRED", "details": "Original annual composites may include post-event imagery"},
    {"check": "Final recovered groups", "status": "NOT_AUTHORIZED", "details": 0},
    {"check": "Model retraining", "status": "NOT_AUTHORIZED", "details": "Rerun full LR/RF/XGBoost benchmarking after all data gates"},
])
gate.to_csv(OUTPUT_DIR / "stage13b5_decision_gate.csv", index=False)
print(gate.to_string(index=False))
print(json.dumps(summary, indent=2))

## PART E — NEXT ACTIONS

### Interpretation of outputs

- `candidate_spatial_and_feature_audit.csv`: all candidate QC results plus nearest positive distance; **distance-to-nearest-case is an audit, not the original spatial approval**.
- `provisional_unique_control_assignments.csv`: unique background controls tentatively assigned to matched groups.
- `replacement_slot_final_screen.csv`: unresolved slots, including `POS_0131` if no alternatives were found.
- `PROVISIONAL_recovered_temporal_matched_groups_NOT_FOR_TRAINING.csv`: a convenient **inspection artifact**, not an approved training dataset.
- `stage13b5_summary.json`, `stage13b5_decision_gate.csv`: whether required scientific gates are resolved.

### Before final model retraining

1. Upload the original Stage 04/05A pseudo-absence generation notebook or its specific exclusion masks and buffer setting. **Do not invent a buffer size**. The source generation may have already screened candidate points, but the rule and coverage must still be audited.
2. Independently verify historic reference-date and annual land-cover leakage policy, including the *existing* 121-case groups.
3. Inspect replacement land-cover encoding consistency with the Stage 09 `landcover_harmonised` schema. Dynamic World `PRE_EVENT_DW_3PLUS_PROVISIONAL` is **not** automatically equivalent to the original `HIGH` reliability flag.
4. Only after approval, regenerate model-ready temporal subsets and **rerun** Logistic Regression, Random Forest, and XGBoost with the same 5-outer / 3-inner spatial CV; record all metrics, fold variability and static-only ablation on matched data.
5. Do not overwrite Stage 12 models before the new results have been validated.

**Important:** Creating a larger provisional dataset does not demonstrate improved predictive validity by itself.